# FT 3B Solo — SVAMP Direct Answer Fine-Tuning

**Purpose:** Fine-tune Qwen2.5-3B-Instruct to answer SVAMP arithmetic questions **directly** (no guide, no pipeline). Then evaluate accuracy on N=300 questions at seed=42 — the same questions used in the main paper.

**Why this matters:** The guided pipeline (FT 3B guide + 1.5B×5) costs 10.5B parameter-passes. This experiment costs 3B parameter-passes (one forward pass). If FT 3B solo matches or approaches 61.7% at 3× less compute, the pipeline's architecture claim needs rethinking.

| Condition | Compute | Expected |
|---|---|---|
| Baseline (1.5B×5) | 7.5B pp | 40.3% (confirmed) |
| **FT 3B Solo (this run)** | **3.0B pp** | **TBD** |
| Guided pipeline | 10.5B pp | 61.7% (confirmed) |

> Seed=42 · N=300 · Same question split as original paper run

In [1]:
# CELL 1 — Install
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q peft==0.12.0
# !pip install -q datasets==2.20.0
# !pip install -q trl==0.9.6
# !pip install -q huggingface_hub
print("Done.")

Done.


In [ ]:
# CELL 2 — Login
from huggingface_hub import login
login("")  # paste your HF token
print("Login done")

Login done


In [3]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 21.1 MB/s eta 0:00:00


In [4]:
# CELL 3 — Imports
import os, json, re, random, time, math
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from tqdm.notebook import tqdm

OUTPUT_DIR = "/content/svamp_ft3b_solo"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Output: {OUTPUT_DIR}")

GPU: Tesla T4
VRAM: 15.6 GB
Output: /content/svamp_ft3b_solo


In [5]:
# CELL 4 — Config
# CRITICAL: eval seed and N must match original paper run exactly.
CONFIG = {
    "model_name"        : "Qwen/Qwen2.5-1.5B-Instruct",
    "dataset_name"      : "ChilleD/SVAMP",
    "eval_seed"         : 42,
    "eval_n"            : 300,
    "train_split_seed"  : 0,       # separate seed for train split
    "max_train_samples" : 700,     # SVAMP train set ~800 examples
    "lora_r"            : 16,
    "lora_alpha"        : 32,
    "lora_dropout"      : 0.05,
    "learning_rate"     : 2e-4,
    "num_epochs"        : 3,
    "batch_size"        : 4,
    "grad_accum"        : 4,
    "max_seq_length"    : 256,
    "eval_temperature"  : 0.0,     # greedy — single forward pass
    "max_new_tokens"    : 64,
    "results_file"      : f"{OUTPUT_DIR}/results.jsonl",
    "checkpoint_file"   : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"        : 50,
}
print("Config ready. eval_seed=42, eval_n=300 — matches paper.")

Config ready. eval_seed=42, eval_n=300 — matches paper.


In [6]:
# CELL 5 — Load and split SVAMP
print("Loading SVAMP...")
raw_ds = load_dataset(CONFIG["dataset_name"])

# Build full question list
def make_question(item):
    body = item.get("Body","").strip()
    question = item.get("Question","").strip()
    answer = str(item.get("Answer","")).strip()
    full_q = f"{body} {question}".strip()
    return {"question": full_q, "answer": answer}

all_data = [make_question(x) for x in raw_ds["train"]]
# Some datasets use 'test' split
if "test" in raw_ds:
    all_data += [make_question(x) for x in raw_ds["test"]]

print(f"Total SVAMP examples: {len(all_data)}")

# Eval split: seed=42, N=300 — must match paper
random.seed(CONFIG["eval_seed"])
eval_data = random.sample(all_data, CONFIG["eval_n"])
eval_ids  = set(id(x) for x in eval_data)

# Train split: remaining examples, different seed
train_candidates = [x for x in all_data if x not in eval_data]
random.seed(CONFIG["train_split_seed"])
if len(train_candidates) > CONFIG["max_train_samples"]:
    train_data = random.sample(train_candidates, CONFIG["max_train_samples"])
else:
    train_data = train_candidates

print(f"Eval set : {len(eval_data)} questions (seed=42 — paper split)")
print(f"Train set: {len(train_data)} questions")
print(f"Overlap check: {len(set(x["question"] for x in eval_data) & set(x["question"] for x in train_data))} shared questions (should be 0)")

Loading SVAMP...


README.md:   0%|          | 0.00/675 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/111k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/54.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/700 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Total SVAMP examples: 1000
Eval set : 300 questions (seed=42 — paper split)
Train set: 700 questions
Overlap check: 0 shared questions (should be 0)


In [7]:
# CELL 6 — Format training data as SFT prompt
# The model is trained to produce only the numeric answer — no explanation.
# This matches the evaluation format (extract a number from output).

SYSTEM_PROMPT = (
    "You are a precise arithmetic solver.\n"
    "Read the problem carefully and output only the numeric answer.\n"
    "Do not show any working. Output the number only."
)

def format_sft(item, tokenizer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Problem: {item['question']}"},
        {"role": "assistant", "content": str(item["answer"])},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

print("SFT format ready.")
print("Example:\n")
print(f"  Q: {train_data[0]["question"]}")
print(f"  A: {train_data[0]["answer"]}")

SFT format ready.
Example:

  Q: Marco and his dad went strawberry picking. Marco's dad's strawberries weighed 11 pounds. If together their strawberries weighed 30 pounds. How much did Marco's strawberries weigh?
  A: 19


In [8]:
# CELL 7 — Load tokenizer
print(f"Loading tokenizer: {CONFIG["model_name"]}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer ready.")

Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer ready.


In [9]:
# CELL 8 — Prepare HF Dataset for SFTTrainer
from datasets import Dataset

train_formatted = [format_sft(x, tokenizer) for x in train_data]
hf_train = Dataset.from_list(train_formatted)

print(f"Training examples: {len(hf_train)}")
print(f"Sample text (first 200 chars): {hf_train[0]["text"][:200]}")

Training examples: 700
Sample text (first 200 chars): <|im_start|>system
You are a precise arithmetic solver.
Read the problem carefully and output only the numeric answer.
Do not show any working. Output the number only.<|im_end|>
<|im_start|>user
Probl


In [10]:
# CELL 9 — Load base model
print(f"Loading: {CONFIG["model_name"]}")
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.config.use_cache = False
base_model.enable_input_require_grads()
vram = torch.cuda.memory_allocated()/1e9
print(f"VRAM after load: {vram:.2f}GB")

Loading: Qwen/Qwen2.5-1.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

VRAM after load: 1.50GB


In [11]:
# CELL 10 — Attach LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print("LoRA attached.")

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
LoRA attached.


In [12]:
# CELL 11 — Training
print(f"Starting fine-tuning: {CONFIG["num_epochs"]} epochs, {len(hf_train)} examples")

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    dataloader_num_workers=0,
    seed=42,
)



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting fine-tuning: 3 epochs, 700 examples


In [13]:
trainer = SFTTrainer(
    model=model,
    train_dataset=hf_train,
    processing_class=tokenizer,
    args=training_args,
)

t0 = time.time()
trainer.train()
print(f"\nTraining done in {(time.time()-t0)/60:.1f} min.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,1.479896
40,0.534166
60,0.397128
80,0.359314
100,0.290047
120,0.255803



Training done in 2.7 min.


In [14]:
# CELL 12 — Save fine-tuned model
ft_model_path = f"{OUTPUT_DIR}/ft_model"
trainer.save_model(ft_model_path)
tokenizer.save_pretrained(ft_model_path)
print(f"Model saved to {ft_model_path}")

Model saved to /content/svamp_ft3b_solo/ft_model


In [15]:
# CELL 13 — Answer extraction
def extract_number(text):
    text = text.strip()
    # 1. First number on first line
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines:
        m = re.match(r"^-?\d+(?:\.\d+)?$", lines[0])
        if m: return lines[0]
    # 2. 'answer is X'
    m = re.search(r"(?:answer\s+is|=)\s*(-?\d+(?:\.\d+)?)", text, re.IGNORECASE)
    if m: return m.group(1)
    # 3. Last standalone number
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums: return nums[-1]
    return ""

def normalize(s):
    try:
        v = float(str(s).replace(',','').strip())
        return str(int(v)) if v == int(v) else str(v)
    except:
        return str(s).strip().lower()

# Quick tests
_t = [("42","42"),("The answer is 15","15"),("3.0","3"),("\n\n7","7")]
ok = all(normalize(extract_number(t))==normalize(e) for t,e in _t)
print("Extractor:", "ALL PASSED" if ok else "FAIL")

Extractor: ALL PASSED


In [16]:
# CELL 14 — Load fine-tuned model for eval
# Reload fresh to avoid any training state contamination
del model, base_model, trainer
torch.cuda.empty_cache()

from peft import PeftModel

eval_base = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], torch_dtype=torch.float16, device_map="auto"
).eval()
ft_model = PeftModel.from_pretrained(eval_base, ft_model_path).eval()
print("Fine-tuned model loaded for evaluation.")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fine-tuned model loaded for evaluation.
VRAM: 3.22GB


In [17]:
# CELL 15 — Single-pass evaluation function
# FT 3B Solo uses ONE forward pass (greedy, T=0) — not 5 votes.
# Compute = 3B × 1 = 3.0B pp  (vs guided = 10.5B pp)

EVAL_SYSTEM = (
    "You are a precise arithmetic solver.\n"
    "Read the problem carefully and output only the numeric answer.\n"
    "Do not show any working. Output the number only."
)

def run_ft_solo(question):
    messages = [
        {"role": "system",    "content": EVAL_SYSTEM},
        {"role": "user",      "content": f"Problem: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    device = next(ft_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False,          # greedy — deterministic
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True).strip()

print("Eval function ready. Single greedy pass — 3.0B pp per question.")

Eval function ready. Single greedy pass — 3.0B pp per question.


In [18]:
# CELL 16 — Verification run (20 questions)
# Sanity check before full eval
v_correct = 0; v_empty = 0
print("Verification: 20 questions...")
for item in eval_data[:20]:
    raw = run_ft_solo(item["question"])
    pred = normalize(extract_number(raw))
    gt   = normalize(item["answer"])
    if not pred: v_empty += 1
    if pred == gt: v_correct += 1

print(f"Verification: {v_correct}/20 = {v_correct/20*100:.0f}% correct")
print(f"Empty answers: {v_empty}/20")
print(f"Expected range: 30-70% (model just fine-tuned)")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Verification: 20 questions...
Verification: 11/20 = 55% correct
Empty answers: 0/20
Expected range: 30-70% (model just fine-tuned)


In [19]:
# CELL X — Self Consistency (5 votes majority)

from collections import Counter

def run_ft_self_consistency(question, votes=5):

    messages = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user", "content": f"Problem: {question}"},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    device = next(ft_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    answers = []

    for _ in range(votes):
        with torch.no_grad():
            out = ft_model.generate(
                **inputs,
                max_new_tokens=CONFIG["max_new_tokens"],
                do_sample=True,          # enable sampling
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_toks = out[0][inputs["input_ids"].shape[1]:]
        text = tokenizer.decode(new_toks, skip_special_tokens=True).strip()

        num = normalize(extract_number(text))
        if num != "":
            answers.append(num)

    if len(answers) == 0:
        return ""

    vote = Counter(answers)
    majority_answer = vote.most_common(1)[0][0]

    return majority_answer

In [20]:
# CELL 17 — Full evaluation (N=300)
# ~5-10 minutes on T4 (single pass, no voting)
print(f"Evaluating {CONFIG["eval_n"]} questions...")
print(f"Compute: 3.0B pp per question (1 forward pass)")
print("-"*60)

results = []; start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f: ck = json.load(f)
    start_idx = ck.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed from {start_idx}")

t0 = time.time()
for idx in tqdm(range(start_idx, len(eval_data)), desc="FT-3B-Solo"):
    item = eval_data[idx]
    try:
        raw  = run_ft_solo(item["question"])
        pred = normalize(extract_number(raw))
        gt   = normalize(item["answer"])
        results.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"],
            "raw_output": raw, "final_answer": pred,
            "correct": (pred == gt), "empty": (pred == ""),
        })
    except Exception as e:
        results.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"], "raw_output": "",
            "final_answer": "", "correct": False, "empty": True,
            "error": str(e)
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in results: f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx+1}, f)
        acc = sum(r["correct"] for r in results) / len(results) * 100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:3d}] acc={acc:.1f}%  ({mins:.1f}min)")

with open(CONFIG["results_file"], "w") as f:
    for r in results: f.write(json.dumps(r) + "\n")

n_correct = sum(r["correct"] for r in results)
n_empty   = sum(r["empty"]   for r in results)
print(f"\n=== FINAL RESULT ===")
print(f"  FT 3B Solo:      {n_correct}/{len(results)} = {n_correct/len(results)*100:.1f}%")
print(f"  Empty answers:   {n_empty}")
print(f"  Compute:         3.0B pp (1 forward pass)")

Evaluating 300 questions...
Compute: 3.0B pp per question (1 forward pass)
------------------------------------------------------------


FT-3B-Solo:   0%|          | 0/300 [00:00<?, ?it/s]

  [ 50] acc=60.0%  (0.2min)
  [100] acc=51.0%  (0.5min)
  [150] acc=51.3%  (0.7min)
  [200] acc=49.5%  (0.9min)
  [250] acc=46.4%  (1.2min)
  [300] acc=47.3%  (1.5min)

=== FINAL RESULT ===
  FT 3B Solo:      142/300 = 47.3%
  Empty answers:   0
  Compute:         3.0B pp (1 forward pass)


In [21]:
# CELL 17 — Full evaluation (Self Consistency 5 vote)

print(f"Evaluating {CONFIG['eval_n']} questions...")
print("Compute: 15.0B pp per question (5 voting passes)")
print("-"*60)

results = []
start_idx = 0

# ----- CHECKPOINT LOGIC -----
if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ck = json.load(f)

    start_idx = ck.get("last_index", 0)

    # If previous run finished, start a new run instead of skipping everything
    if start_idx >= len(eval_data):
        print("Previous run already completed — starting a new evaluation run.")
        start_idx = 0
        results = []
    else:
        if os.path.exists(CONFIG["results_file"]):
            with open(CONFIG["results_file"]) as f:
                results = [json.loads(l) for l in f if l.strip()]
        print(f"Resuming from index {start_idx}")

t0 = time.time()

for idx in tqdm(range(start_idx, len(eval_data)), desc="FT-3B-SC(5)"):

    item = eval_data[idx]

    try:
        raw  = run_ft_self_consistency(item["question"], votes=5)
        pred = normalize(extract_number(raw))
        gt   = normalize(item["answer"])

        results.append({
            "idx": idx,
            "question": item["question"],
            "gt_answer": item["answer"],
            "raw_output": raw,
            "final_answer": pred,
            "correct": (pred == gt),
            "empty": (pred == "")
        })

    except Exception as e:
        results.append({
            "idx": idx,
            "question": item["question"],
            "gt_answer": item["answer"],
            "raw_output": "",
            "final_answer": "",
            "correct": False,
            "empty": True,
            "error": str(e)
        })

    if (idx + 1) % CONFIG["save_every"] == 0:

        with open(CONFIG["results_file"], "w") as f:
            for r in results:
                f.write(json.dumps(r) + "\n")

        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx+1}, f)

        acc = sum(r["correct"] for r in results) / len(results) * 100
        mins = (time.time()-t0)/60

        print(f"  [{idx+1:3d}] acc={acc:.1f}%  ({mins:.1f} min)")


# ----- FINAL SAVE -----
with open(CONFIG["results_file"], "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

n_correct = sum(r["correct"] for r in results)
n_empty   = sum(r["empty"] for r in results)

print("\n=== FINAL RESULT ===")
print(f"  FT 3B (5 Vote): {n_correct}/{len(results)} = {n_correct/len(results)*100:.1f}%")
print(f"  Empty answers:  {n_empty}")
print("  Compute:        15.0B pp (5 voting passes)")

Evaluating 300 questions...
Compute: 15.0B pp per question (5 voting passes)
------------------------------------------------------------
Previous run already completed — starting a new evaluation run.


FT-3B-SC(5):   0%|          | 0/300 [00:00<?, ?it/s]

  [ 50] acc=56.0%  (1.1 min)
  [100] acc=48.0%  (2.2 min)
  [150] acc=50.0%  (3.4 min)
  [200] acc=48.0%  (4.6 min)
  [250] acc=45.2%  (5.9 min)
  [300] acc=46.0%  (7.1 min)

=== FINAL RESULT ===
  FT 3B (5 Vote): 138/300 = 46.0%
  Empty answers:  0
  Compute:        15.0B pp (5 voting passes)


In [22]:
# CELL 18 — Final comparison table
ft_acc = sum(r["correct"] for r in results) / len(results) * 100

print("="*65)
print("SVAMP — FULL COMPUTE-ACCURACY COMPARISON")
print("="*65)
print(f"  Condition              | Compute   | Accuracy")
print(f"  -----------------------|-----------|----------")
print(f"  Baseline (1.5B×5)      | 7.5B pp   | 40.3%   (confirmed paper)")
print(f"  FT 3B Solo (this run)  | 3.0B pp   | {ft_acc:.1f}%")
print(f"  Guided pipeline        | 10.5B pp  | 61.7%   (confirmed paper)")
print(f"  Ceiling (3B×5 untuned) | 15.0B pp  | 30.7%   (confirmed paper)")
print()
gap = 61.7 - ft_acc
print(f"  Guided vs FT Solo: {gap:+.1f} pts")
print()
if ft_acc >= 60.0:
    print("  VERDICT: FT Solo matches guided pipeline.")
    print("  The architecture overhead is not justified on SVAMP.")
elif ft_acc >= 50.0:
    print(f"  VERDICT: FT Solo is competitive. Guided adds {gap:.1f} pts at 3.5× compute.")
    print("  Discuss whether the accuracy premium justifies the cost.")
else:
    print(f"  VERDICT: Guided pipeline adds {gap:.1f} pts over FT Solo.")
    print("  The guided architecture provides value beyond fine-tuning alone.")

SVAMP — FULL COMPUTE-ACCURACY COMPARISON
  Condition              | Compute   | Accuracy
  -----------------------|-----------|----------
  Baseline (1.5B×5)      | 7.5B pp   | 40.3%   (confirmed paper)
  FT 3B Solo (this run)  | 3.0B pp   | 46.0%
  Guided pipeline        | 10.5B pp  | 61.7%   (confirmed paper)
  Ceiling (3B×5 untuned) | 15.0B pp  | 30.7%   (confirmed paper)

  Guided vs FT Solo: +15.7 pts

  VERDICT: Guided pipeline adds 15.7 pts over FT Solo.
  The guided architecture provides value beyond fine-tuning alone.
